# 03. Обучение и сравнение моделей
Используется тот же модуль, что и в CLI tsprep-comparison. $\beta$ выбирается на проверочной выборке для первой инициализации и фиксируется для остальных повторов. Полный запуск включает 144 выбранные модели и 24 дополнительных кандидата $\beta$, до 30 эпох на обучение.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from datetime import datetime, timezone
from tsprep.comparison_experiment import run_experiment
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 200
SEEDS = [42, 43, 44]
DATASETS = ["synthetic", "aapl", "btc"]
EXPERIMENT_DIR = ROOT / "results" / datetime.now(timezone.utc).strftime("experiment_%Y%m%dT%H%M%S%fZ")
print(EXPERIMENT_DIR, DEVICE)


In [ ]:
# Настройки, данные, нормализатор и веса сохраняются в папке эксперимента.
# Для продолжения укажите прежний EXPERIMENT_DIR с теми же настройками и исходниками.
run_experiment(ROOT, EXPERIMENT_DIR, epochs=EPOCHS, seeds=SEEDS, datasets=DATASETS, device=DEVICE, data_seed=2026, deterministic=True)


In [ ]:
import pandas as pd
runs = pd.read_csv(EXPERIMENT_DIR / "comparison_runs.csv")
runs.groupby(["dataset", "model"])[["mse", "forecast_mae", "forecast_vs_mean", "forecast_vs_median", "forecast_vs_raw", "forecast_vs_pca", "mse_vs_pca"]].agg(["mean", "std"])


Среднее и стандартное отклонение по инициализациям и периодам показывают разброс результатов. Для доверительных интервалов учитывается временная зависимость. Положительные forecast_vs_mean / mse_vs_pca означают проигрыш методу сравнения.


In [ ]:
ALLOW_DOWNSTREAM_SOURCE_MISMATCH = False  # New study uses the current evaluator.

## Прогноз, классификация и методы сравнения
Сводки разделены по набору данных и тестовому периоду. Среднее и стандартное отклонение показывают разброс между инициализациями, а не между независимыми экономическими историями. Каждый метод сравнения добавлен один раз для периода. `count` показывает число доступных измерений; пустая метрика не означает нулевое качество.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print
from tsprep.evaluation.experiment_results import load_experiment_tables, task_summary
from tsprep.evaluation.visualize import plot_task_metric

def show_task(frame, metrics):
    if frame.empty:
        print("Нет данных для этой задачи. Отсутствующая метрика не заменяется нулём.")
    else:
        display(task_summary(frame, metrics))

In [ ]:
experiment_tables = load_experiment_tables(EXPERIMENT_DIR, allow_source_mismatch=ALLOW_DOWNSTREAM_SOURCE_MISMATCH)
display(experiment_tables["availability"])
display(experiment_tables["support"])

### Прогнозирование
Ridge обучается на эмбеддингах обучающей выборки; коэффициент регуляризации выбирается на проверочной. Цель задаётся одним будущим преобразованным наблюдением в t+H, не ценой и не суммой доходностей. Для сравнения используются среднее, медиана, последнее наблюдение, Ridge по исходному окну и PCA+Ridge. Отрицательная `delta_mae_vs_*` означает преимущество перед соответствующим методом.


In [ ]:
show_task(experiment_tables["forecast"], ["mae", "r2", "validation_mae", "delta_mae_vs_mean",
                                    "delta_mae_vs_median", "delta_mae_vs_raw", "delta_mae_vs_pca"])
plot_task_metric(experiment_tables["forecast"], "mae", "Forecast MAE")
plt.show()

### Классификация и разделимость режимов
Бинарная классификация выполняется только при наличии нужных меток. Для трёх синтетических режимов считается silhouette. Классификация сравнивается с PCA и прогнозом самого частого обучающего класса. USREC задаёт ретроспективную метку периода, не публикацию в реальном времени. При одном классе в тесте AUC и часть других показателей не определены. Silhouette по истинным меткам и кластеризация KMeans решают разные диагностические задачи.


In [ ]:
show_task(experiment_tables["classification"], ["accuracy", "balanced_accuracy", "f1", "precision", "recall",
                                          "pr_auc", "roc_auc", "false_positive_rate", "n_positive", "n_negative"])
show_task(experiment_tables["clustering"], ["silhouette", "n_clusters"])

### Аномалии: окна и концы окон
Оценка всего окна = MSE + λ·KL, оценка конца окна = MSE последней точки. Порог выбирается только на проверочной выборке. Квантиль смешанной проверочной выборки не задаёт 5% FPR на нормальных точках. Для сравнения используются PCA и величина стандартизованного входа (raw magnitude). Метрики всего окна и последней точки считаются отдельно. Число эпизодов относится только к наблюдаемым концам окон с заданным шагом.


In [ ]:
show_task(experiment_tables["anomaly"], ["precision", "recall", "f1", "pr_auc", "false_positive_rate", "threshold",
    "positive_rate_test", "predicted_positive_rate", "n_positive", "n_negative", "n_events",
    "detected_events", "event_recall", "false_positive_points"])

### Восстановление, чистый эталон и латентное пространство
Чистый эталон доступен только на синтетике. Сравниваются MSE и MAE к эталону, исходный вход и PCA. Сохранённые ошибки к эталону усредняют все элементы перекрывающихся окон; разбор концов окон показан отдельно. Effective rank, active units и KL используются для диагностики. Отношение дисперсий зависит от масштаба и не измеряет информацию.


In [ ]:
show_task(experiment_tables["reconstruction"], ["mse", "mae", "r2"])
show_task(experiment_tables["clean"], ["mse", "mae", "r2", "delta_mse_vs_input", "delta_mae_vs_input"])
show_task(experiment_tables["latent"], ["effective_rank", "active_units", "kl", "variance_ratio"])

На синтетике с тремя режимами оценивается разделимость режимов, на реальных CSV отсутствуют метки аномалий. В этом ноутбуке модели обучаются по observed → observed. Сравнение обучения на observed/clean находится в ноутбуке 05.
